In [ ]:
#SETUP DOCKER+LANGFUSE

#!cd ~/Desktop/Mini-Project-LangFuse/langfuse
#!docker compose up -d
#http://localhost:3000/

In [ ]:
#SETUP LLM+LANGFUSE

#Import keys
import os
import json
from dotenv import load_dotenv
load_dotenv()

#Set up the LLM client
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY_1"),
    base_url="https://api.groq.com/openai/v1"
)

#Connect LangFuse
from langfuse import observe, Langfuse
langfuse = Langfuse()

#Models
BASELINE_MODEL = "llama-3.1-8b-instant"
NEW_MODEL = "llama-3.3-70b-versatile"

In [ ]:
#SIMPLE CHATBOT

#Prompt
SYSTEM_PROMPT = """You are a helpful healthcare assistant. You provide accurate, 
general health information to help users understand medical topics.

Rules:
- Always recommend consulting a doctor for specific medical advice.
- Be clear and concise.
- If you don't know something, say so honestly.
- Never diagnose conditions or prescribe medication.
"""

#ChatBot
@observe()
def healthcare_chatbot(user_question, model=BASELINE_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_question}
        ]
    )
    return response.choices[0].message.content

In [ ]:
#LLM AS A JUDGE

#Prompt
JUDGE_PROMPT = """You are an expert evaluator for a healthcare chatbot. 
Compare the actual answer to the expected answer and score the actual answer.

Question: {question}

Expected Answer: {expected_answer}

Actual Answer: {actual_answer}

Score the actual answer from 1 to 5:
1 = Completely wrong or harmful
2 = Mostly incorrect or missing key information
3 = Partially correct but incomplete
4 = Mostly correct with minor omissions
5 = Fully correct and complete

Respond with ONLY a JSON object in this exact format, nothing else:
{{"score": <number>, "reasoning": "<brief explanation>"}}
"""

#Judge
@observe()
def judge_answer(question, expected_answer, actual_answer, model=BASELINE_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": JUDGE_PROMPT.format(
                question=question,
                expected_answer=expected_answer,
                actual_answer=actual_answer
            )}
        ]
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
#GOLDEN DATASET (20 Healthcare question/answer pairs across 7 categories)

with open("../Data/golden_dataset.json", "r") as f:
    golden_dataset = json.load(f)

print(f"Loaded {len(golden_dataset)} questions")

In [ ]:
#EVALUATE CHATBOT ANSWERS USING LLM AS A JUDGE

def run_evaluation(model_name):
    print(f"\n{'='*50}")
    print(f"Evaluating model: {model_name}")
    print(f"{'='*50}\n")
    
    results = []
    
    for item in golden_dataset:
        print(f"[{item['id']}/20] {item['question'][:55]}...")
        
        #Chatbot answer
        answer = healthcare_chatbot(item["question"], model=model_name)
        
        #Judge the answer
        eval_result = judge_answer(
            item["question"], 
            item["expected_answer"], 
            answer, 
            model=BASELINE_MODEL  
        )

        #Append results
        results.append({
            "id": item["id"],
            "category": item["category"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "actual_answer": answer,
            "score": eval_result["score"],
            "reasoning": eval_result["reasoning"]
        })
        
        print(f"  Score: {eval_result['score']} — {eval_result['reasoning'][:70]}")
    
    avg_score = sum(r["score"] for r in results) / len(results)
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"Average Score: {avg_score:.2f} / 5.00")
    print(f"{'='*50}")
    
    return results, avg_score

baseline_results, baseline_score = run_evaluation(BASELINE_MODEL)
new_results, new_score = run_evaluation(NEW_MODEL)

In [ ]:
#COMPARE MODELS

def compare_results(baseline_results, baseline_score, new_results, new_score):
    baseline_model = BASELINE_MODEL
    new_model = NEW_MODEL
    
    print(f"\n{'='*50}")
    print(f"COMPARISON")
    print(f"{'='*50}")
    print(f"Baseline ({baseline_model}): {baseline_score:.2f}")
    print(f"New model ({new_model}): {new_score:.2f}")
    print(f"Difference: {new_score - baseline_score:+.2f}")
    
    #Per-question breakdown
    print(f"\nPer-question comparison:")
    improved, same, worse = 0, 0, 0
    for b, n in zip(baseline_results, new_results):
        diff = n["score"] - b["score"]
        if diff > 0:
            indicator = "✅"
            improved += 1
        elif diff < 0:
            indicator = "❌"
            worse += 1
        else:
            indicator = "➡️"
            same += 1
        print(f"  {indicator} Q{b['id']}: {b['score']}→{n['score']} ({diff:+d}) {b['question'][:50]}")
    
    #Per-category breakdown
    from collections import defaultdict
    baseline_by_cat = defaultdict(list)
    new_by_cat = defaultdict(list)
    for b, n in zip(baseline_results, new_results):
        baseline_by_cat[b["category"]].append(b["score"])
        new_by_cat[n["category"]].append(n["score"])
    
    print(f"\nPer-category comparison:")
    for cat in baseline_by_cat:
        b_avg = sum(baseline_by_cat[cat]) / len(baseline_by_cat[cat])
        n_avg = sum(new_by_cat[cat]) / len(new_by_cat[cat])
        diff = n_avg - b_avg
        indicator = "✅" if diff > 0 else "❌" if diff < 0 else "➡️"
        print(f"  {indicator} {cat}: {b_avg:.2f}→{n_avg:.2f} ({diff:+.2f})")
    
    #Summary
    print(f"\nSummary: {improved} improved, {same} same, {worse} worse")xw

compare_results(baseline_results, baseline_score, new_results, new_score)